[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/research_assistant/plain_python.ipynb)

# Research assistant — plain Python

**Task.** Answer which interventions reduce household food waste using only a bounded, safety-screened evidence catalogue.

**Read this notebook as:** choose a provider → load evidence → declare typed boundaries → plan, retrieve, synthesise and critique.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "research-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
API_KEY_ENVIRONMENT_VARIABLE = "GEMINI_API_KEY"
SAVE_API_CREDENTIAL = True  # True stores it locally after the first hidden prompt
API_CREDENTIAL_FILE = Path(".private/gemini_api_key")
# Mock runtime is under one minute on CPU; local and API runs can be slower.

if MODEL_PROVIDER not in {"mock", "local", "api"}:
    raise ValueError("MODEL_PROVIDER must be mock, local or api")
if MODEL_PROVIDER == "local" and not LOCAL_MODEL_PATH:
    raise RuntimeError("Set AGENTIC_TUTORIAL_LOCAL_MODEL_PATH for local execution")
if MODEL_PROVIDER == "api":
    config_root = Path.cwd()
    while config_root.parent != config_root and not (config_root / "pyproject.toml").exists():
        config_root = config_root.parent
    credential_path = config_root / API_CREDENTIAL_FILE
    credential = os.getenv(API_KEY_ENVIRONMENT_VARIABLE, "")
    if not credential and credential_path.is_file():
        credential = credential_path.read_text(encoding="utf-8").strip()
    if not credential:
        credential = getpass(
            "Paste Gemini API key into the hidden input prompt, then press Enter: "
        ).strip()
    if not credential:
        raise RuntimeError(
            "No Gemini key was entered. Re-run this cell, use the hidden input prompt "
            "shown by Jupyter/VS Code, paste the key, and press Enter."
        )
    if SAVE_API_CREDENTIAL and not credential_path.is_file():
        credential_path.parent.mkdir(parents=True, exist_ok=True)
        credential_path.write_text(credential + "\n", encoding="utf-8")
        credential_path.chmod(0o600)
    os.environ[API_KEY_ENVIRONMENT_VARIABLE] = credential
    del credential

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.12.5"], check=True)

In [ ]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.safety import (  # noqa: E402
    PolicyOutcome,
    SafetyEngine,
    SafetyPolicy,
    UntrustedContent,
)
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture_path = ROOT / "data/research_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
RESEARCH_QUESTION = "Which interventions reduce household food waste, and under what conditions?"

## Visible workflow
The model proposes queries, claims and synthesis. Python searches the bounded catalogue, treats passages as untrusted data, validates every claim/source pair, critiques once, and stops after four model calls. Empty or unsupported evidence returns abstention.

In [ ]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: Literal["supporting", "conflicting"]


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: Literal["qualified_answer", "abstention"]


class CriticOutput(Strict):
    accepted: bool
    issues: tuple[str, ...] = ()


Claim.model_rebuild(_types_namespace={"Literal": Literal})
Ledger.model_rebuild(_types_namespace={"Claim": Claim})
Synthesis.model_rebuild(_types_namespace={"Literal": Literal})


def model():
    model_names = {
        "mock": MOCK_MODEL_NAME,
        "local": LOCAL_MODEL_NAME,
        "api": API_MODEL_NAME,
    }
    local_model_path = Path(LOCAL_MODEL_PATH) if LOCAL_MODEL_PATH else None
    if local_model_path is not None and not local_model_path.is_absolute():
        local_model_path = ROOT / local_model_path
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=local_model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


def search(query):
    terms = set(query.casefold().split())
    return [
        r
        for r in catalogue["records"]
        if terms & set((r["title"] + " " + r["passage"]).casefold().split())
    ]


async def propose(client, schema, prompt):
    response = await client.generate([user(prompt)], response_schema=schema)
    return schema.model_validate(response.structured_output)

In [ ]:
async def run_research():
    client = model()
    trace = []
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))
    plan = await propose(
        client,
        QueryPlan,
        f"For this task only: {RESEARCH_QUESTION} Return focused searches covering "
        "smaller plates, meal planning, and information-only campaigns.",
    )
    trace.append({"event": "plan", "queries": plan.queries})
    retrieved = {r["source_id"]: r for q in plan.queries for r in search(q)}
    assessments = {
        sid: safety.inspect_retrieved_content(UntrustedContent(source_id=sid, text=r["passage"]))
        for sid, r in retrieved.items()
    }
    safe_records = {
        sid: r
        for sid, r in retrieved.items()
        if assessments[sid].decision.outcome in {PolicyOutcome.ALLOW, PolicyOutcome.TRANSFORM}
        and not assessments[sid].indicators
    }
    trace.append(
        {
            "event": "retrieve",
            "source_ids": sorted(retrieved),
            "isolated": [sid for sid, a in assessments.items() if a.indicators],
        }
    )
    ledger = await propose(
        client,
        Ledger,
        f"Extract one verbatim claim per record from {safe_records}. Use each record's "
        "source_id. Use stance supporting for a reported reduction and conflicting for "
        "inconsistent effects.",
    )
    valid_claims = tuple(
        c
        for c in ledger.claims
        if c.source_id in safe_records
        and c.claim.casefold().split()[0] in safe_records[c.source_id]["passage"].casefold()
    )
    trace.append(
        {
            "event": "ledger",
            "supporting": [c.source_id for c in valid_claims if c.stance == "supporting"],
            "conflicting": [c.source_id for c in valid_claims if c.stance == "conflicting"],
        }
    )
    if not valid_claims:
        return {"outcome": "abstention", "trace": trace, "usage": {"model_calls": 2}}
    answer = await propose(
        client,
        Synthesis,
        f"Synthesise only these validated claims with conditions and conflicts: {valid_claims}. "
        "Cite their source_ids and use outcome qualified_answer.",
    )
    known = set(safe_records)
    citations_valid = set(answer.source_ids).issubset(known)
    critic_output = await propose(
        client,
        CriticOutput,
        f"Accept only if supported, qualified and cited: {answer.model_dump()}",
    )
    critic = CriticDecision(
        accepted=critic_output.accepted,
        issues=critic_output.issues,
    )
    outcome = answer.outcome if citations_valid and critic.accepted else "abstention"
    trace.append(
        {"event": "critic", "accepted": critic.accepted, "citations_valid": citations_valid}
    )
    return {
        "outcome": outcome,
        "answer": answer.model_dump(),
        "claim_ledger": [c.model_dump() for c in valid_claims],
        "source_provenance": sorted(answer.source_ids),
        "execution_provenance": {
            "provider": MODEL_PROVIDER,
            "fixture_version": fixture["fixture_version"],
        },
        "trace": trace,
        "termination": "criteria_met",
        "usage": {"model_calls": 4, "tool_calls": len(plan.queries)},
    }


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_research()
second = await run_research() if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": len(first["claim_ledger"]) == 3,
    "trajectory": len(first["trace"]) == 4 and first["usage"]["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["source_provenance"],
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "result": first,
    "resource_report": first["usage"],
    "fallback": "abstain when no validated claims remain",
}

## Limitations
The tiny catalogue cannot establish external validity, lexical claim checks are deliberately conservative, and deterministic fixtures measure orchestration rather than real-model research quality.